# Tempo Run 2025 — Setup Colab

Mục đích: setup môi trường, clone repo, cài deps, tạo `.env`.

**Prerequisite** (làm 1 lần trên máy local trước):
- Upload data lên Hugging Face private dataset — xem `COLAB_GUIDE.md` mục "BƯỚC 0a"
- Repo: `ThinhDao/TempoRun2025_UIT` (đã tạo sẵn)
- Tạo HF token write scope: https://huggingface.co/settings/tokens

Chạy lần lượt từng cell. Cell 4 dùng `%%writefile` để tạo `.env` từ nội dung bạn paste vào.

In [ ]:
# 0. Check GPU
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', torch.cuda.get_device_properties(0).total_memory / 1e9)

In [ ]:
# 1. Clone repo
import os
REPO_URL = "https://github.com/TristanDao/finetune_1B_MCQ_VN.git"
REPO_DIR = "/content/finetune_1B_MCQ_VN"

if os.path.isdir(REPO_DIR):
    %cd {REPO_DIR}
    !git pull --rebase --autostash
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

In [ ]:
# 2. Cài deps (đỡ tốn thời gian nếu đã cài)
!pip install -q -e .
!pip install -q -r requirements.txt

## 3. Tạo `.env`

**Bước A**: Mở `.env.example` để xem các biến cần điền.

**Bước B**: Chỉnh nội dung bên dưới (cell `%%writefile .env`) rồi chạy.

Bắt buộc:
- `HF_TOKEN` (write scope, lấy ở https://huggingface.co/settings/tokens)
- `HF_DATASET_REPO` (mặc định: `ThinhDao/TempoRun2025_UIT`)
- `HF_DATASET_FILE` (mặc định: `tempo-run-2025-run-with-ai-break-limits.zip`)
- `HF_REPO` (model repo, tạo khi push ở Bước 7)

Optional:
- `DASHSCOPE_API_KEY` (cho Bước 2 enrichment)
- `DRIVE_ROOT` (cho backup checkpoint)

In [ ]:
%%writefile .env
# === Hugging Face (data + model) ===
HF_TOKEN=hf_your_hf_token
HF_DATASET_REPO=ThinhDao/TempoRun2025_UIT
HF_DATASET_FILE=tempo-run-2025-run-with-ai-break-limits.zip
HF_REPO=your_hf_username/temprun-qwen3-0_6b

# === Alibaba DashScope (optional) ===
DASHSCOPE_API_KEY=sk-your-dashscope-key
DASHSCOPE_BASE_URL=https://dashscope.aliyuncs.com/compatible-mode/v1
DASHSCOPE_MODEL=qwen3-max-preview

# === Google Drive backup ===
DRIVE_ROOT=/content/drive/MyDrive/temprun_runs

In [ ]:
# 4. Verify .env đã load đúng (không in giá trị, chỉ check tồn tại)
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
from temprun.utils import load_env, repo_root
load_env(repo_root() / ".env")
import os
required = ["HF_TOKEN", "HF_DATASET_REPO", "HF_DATASET_FILE", "HF_REPO"]
for k in required:
    v = os.environ.get(k, "")
    print(f"{k:20s} {'OK' if v and 'your' not in v else 'MISSING'}")
print()
print("Optional:")
for k in ["DASHSCOPE_API_KEY", "DRIVE_ROOT"]:
    v = os.environ.get(k, "")
    print(f"{k:20s} {'OK' if v else '(empty - OK)'}")

In [ ]:
# 5. (Tuỳ chọn) Mount Drive để persist checkpoint giữa các session
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_ROOT = os.environ.get("DRIVE_ROOT", "/content/drive/MyDrive/temprun_runs")
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f"DRIVE_ROOT = {DRIVE_ROOT}")

In [ ]:
# 6. Smoke test import
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
from temprun.data import build_rows, stratified_split
from temprun.prompts import build_user_instruction
from temprun.utils import parse_generated, set_seed
print("All imports OK")